# Reading, PA — LVT Model (LYCD: AVM + KNN Moving Window)

**Reform:** 4:1 split-rate (land taxed at 4× the improvement rate), revenue-neutral  
**Levy scope:** City of Reading municipal real estate tax only  
**Land value method:** LYCD — KNN moving window (K=75 nearest improved parcels)  
applied to AVM market values from berks_open_avmkit (valuation date 2026-01-01)  
**Improvement value:** AVM total prediction minus LYCD land  
**Current tax base:** 1994 CAMA assessed values (same as model.ipynb)  
**Data sources:** Berks County GIS / CAMA_Master + berks_open_avmkit main/ensemble AVM

## LYCD method

For each parcel:
1. Find K=75 nearest *improved* parcels that have AVM predictions and positive lot area
2. Compute `median(AVM_total / lot_area_sqft)` over those neighbors → local land $/sqft
3. `lycd_land = local_psf × 0.20 × lot_area` for improved parcels  
   `lycd_land = local_psf × 1.00 × lot_area` for vacant parcels  
4. `lycd_impr = max(0, AVM_total − lycd_land)` (fallback to CAMA for 79 uncovered parcels)

This approach requires no predefined zone boundaries and naturally captures Reading's
land value gradient from the commercial core outward.

In [ ]:
import sys
import concurrent.futures
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from scipy.spatial import cKDTree

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

CITY_NAME = 'reading'
STATE_FIPS = '42'
COUNTY_FIPS = '011'
MODEL_TYPE = 'split_rate:4.0_lycd'
LAND_IMPROVEMENT_RATIO = 4.0
CITY_MILLAGE = 19.217
OFFICIAL_REVENUE = 27_235_265
K_NEIGHBORS = 75

AVM_BASE = Path('C:/projects/berks_open_avmkit/notebooks/pipeline/data/us-pa-berks/out/models')
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

## Section 2 — Load Parcel Data

In [ ]:
PARCEL_PATH = DATA_DIR / 'parcels.gpq'
if not PARCEL_PATH.exists():
    raise FileNotFoundError(
        f"Parcel cache not found at {PARCEL_PATH}.\n"
        "Run model.ipynb first (it creates the cache from berks_open_avmkit)."
    )
gdf = gpd.read_parquet(PARCEL_PATH)
print(f"Loaded {len(gdf):,} Reading parcels from cache")

for col in ['assr_land_value', 'assr_impr_value', 'assr_market_value', 'land_area_sqft']:
    gdf[col] = pd.to_numeric(gdf[col], errors='coerce').fillna(0.0)

print(f"\nland_area_sqft > 0:    {(gdf['land_area_sqft'] > 0).sum():,}")
print(f"land_area_sqft = 0:    {(gdf['land_area_sqft'] <= 0).sum():,}")
print(f"\nValue column medians (1994 CAMA):")
print(gdf[['assr_land_value', 'assr_impr_value', 'assr_market_value']].median().round(0))

## Section 3 — Load AVM Predictions

Merge total-value predictions from the main/ensemble model for each model group.
Priority order (descending quality): residential_sf → residential_mf → commercial → vacant.

In [ ]:
MODEL_GROUPS = ['residential_sf', 'residential_mf', 'commercial', 'vacant']

frames = []
for i, mg in enumerate(MODEL_GROUPS):
    path = AVM_BASE / mg / 'main' / 'ensemble' / 'pred_universe.parquet'
    if path.exists():
        df = pd.read_parquet(path, columns=['key', 'prediction'])
        df['_priority'] = i
        frames.append(df)
        print(f"  {mg}: {len(df):,} county parcels")
    else:
        print(f"  {mg}: not found at {path}")

avm = (
    pd.concat(frames)
    .sort_values('_priority')
    .drop_duplicates('key', keep='first')
    .drop(columns='_priority')
)
print(f"\nAVM predictions (county-wide): {len(avm):,} unique parcels")

gdf = gdf.merge(avm, on='key', how='left')
n_covered = gdf['prediction'].notna().sum()
print(f"Reading parcels with AVM prediction: {n_covered:,} / {len(gdf):,} ({n_covered/len(gdf):.1%})")
print(f"Reading parcels without AVM coverage: {gdf['prediction'].isna().sum():,} (will get KNN land values from neighbors)")

print(f"\nAVM prediction stats (Reading, covered parcels):")
print(gdf['prediction'].describe().round(0))

## Section 4 — Classify Parcels

In [ ]:
gdf['full_exmp'] = (gdf['category_code'] == 'E').astype(int)

def classify_parcel(row):
    cat = str(row.get('category_code', ''))
    luc = str(row.get('luc', ''))
    if cat == 'E':
        return 'Exempt'
    elif cat == 'R':
        if luc.endswith('R') or luc in ['162', '163']:
            return 'Townhome / Rowhouse'
        elif luc[:2] in ['12', '13']:
            return 'Small Multi-Family (2-4 units)'
        elif luc in ['149', '153']:
            return 'Other Residential'
        else:
            return 'Single Family Residential'
    elif cat == 'A':
        return 'Large Multi-Family (5+ units)'
    elif cat in ['C', 'I']:
        luc_prefix = luc[:2]
        if luc_prefix == '42':
            return 'Large Multi-Family (5+ units)'
        elif luc_prefix in ['23', '24']:
            return 'Small Multi-Family (2-4 units)'
        elif cat == 'I' or luc_prefix in ['51', '52', '58', '59']:
            return 'Industrial'
        else:
            return 'Commercial'
    elif cat == 'F':
        return 'Agricultural'
    elif cat in ['UE', 'UT']:
        return 'Other'
    else:
        return 'Other'

gdf['PROPERTY_CATEGORY'] = gdf.apply(classify_parcel, axis=1)
gdf.loc[gdf['assr_impr_value'] <= 0, 'PROPERTY_CATEGORY'] = 'Vacant Land'

print("Property category distribution:")
print(gdf['PROPERTY_CATEGORY'].value_counts())

## Section 5 — Current Tax (1994 CAMA)

Current tax uses the same 1994 CAMA base and 2026 adopted millage as model.ipynb.
The LYCD values affect only the *reform scenario*, not the current tax.

In [ ]:
gdf['millage_rate'] = CITY_MILLAGE

current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='assr_market_value',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)

gap_pct = (current_revenue / OFFICIAL_REVENUE - 1) * 100
print(f"Modeled gross levy:  ${current_revenue:,.0f}")
print(f"Official target:     ${OFFICIAL_REVENUE:,.0f}  (2024 actual scaled to 2026 rate)")
print(f"Gap:                 {gap_pct:+.2f}%")

## Section 6 — LYCD Land Values (KNN Moving Window)

In [ ]:
# Project to EPSG:3857 for metric distances
gdf_m = gdf.to_crs('EPSG:3857')
coords = np.column_stack([gdf_m.geometry.centroid.x, gdf_m.geometry.centroid.y])

# Improved parcels with AVM prediction and positive lot area form the PSF reference set
has_pred = gdf['prediction'].notna() & (gdf['prediction'] > 0)
has_area = gdf['land_area_sqft'] > 0
is_improved = (gdf['assr_impr_value'] > 0) & has_pred & has_area

improved_idx = np.where(is_improved.values)[0]
improved_coords = coords[improved_idx]
improved_psf = (gdf['prediction'].values[improved_idx] / gdf['land_area_sqft'].values[improved_idx])

print(f"Improved parcels in KNN reference set: {len(improved_idx):,}")
print(f"AVM total PSF stats (improved, Reading):")
print(pd.Series(improved_psf).describe().round(2))

# Build KNN tree; K capped at reference set size
K = min(K_NEIGHBORS, len(improved_idx))
tree = cKDTree(improved_coords)
_, nn_idxs = tree.query(coords, k=K, workers=-1)
local_psf = np.median(improved_psf[nn_idxs], axis=1)

# KNN-impute lot area for parcels with zero or null area
land_area = gdf['land_area_sqft'].values.copy()
no_area_mask = land_area <= 0
if no_area_mask.sum() > 0:
    _, area_nn = tree.query(coords[no_area_mask], k=min(5, K))
    ref_areas = gdf['land_area_sqft'].values[improved_idx]
    land_area[no_area_mask] = np.median(ref_areas[area_nn], axis=1)
    print(f"\nKNN-imputed lot area for {no_area_mask.sum():,} parcels with zero area")

# LYCD allocation: 20% for improved, 100% for vacant
land_pct = np.where(is_improved.values, 0.20, 1.00)
gdf['lycd_land_value'] = (local_psf * land_pct * land_area).clip(0)

# Cap at AVM total for improved parcels (land cannot exceed total value)
avm_total = gdf['prediction'].fillna(gdf['lycd_land_value'])
cap_mask = is_improved & (gdf['lycd_land_value'] > avm_total)
n_capped = cap_mask.sum()
gdf.loc[cap_mask, 'lycd_land_value'] = avm_total[cap_mask]
if n_capped > 0:
    print(f"Capped lycd_land_value at AVM total for {n_capped:,} improved parcels")

# Implied improvement = AVM total - LYCD land
# Fallback to 1994 CAMA improvement for the ~79 uncovered parcels
gdf['lycd_impr_value'] = np.where(
    has_pred,
    (gdf['prediction'] - gdf['lycd_land_value']).clip(lower=0),
    gdf['assr_impr_value'].clip(lower=0),
)

taxable_mask = gdf['full_exmp'] == 0
print(f"\nLYCD land base  (taxable parcels): ${gdf.loc[taxable_mask, 'lycd_land_value'].sum()/1e6:.1f}M")
print(f"LYCD impr base  (taxable parcels): ${gdf.loc[taxable_mask, 'lycd_impr_value'].sum()/1e6:.1f}M")
print(f"CAMA land base  (taxable parcels): ${gdf.loc[taxable_mask, 'assr_land_value'].sum()/1e6:.1f}M")
print(f"Ratio LYCD/CAMA land:              "
      f"{gdf.loc[taxable_mask,'lycd_land_value'].sum() / gdf.loc[taxable_mask,'assr_land_value'].sum():.2f}x")

## Section 7 — 4:1 Split-Rate Model (LYCD Base)

In [ ]:
taxable = gdf[gdf['full_exmp'] == 0].copy()
taxable['taxable_land_value'] = taxable['lycd_land_value']
taxable['taxable_improvement_value'] = taxable['lycd_impr_value']

land_millage, improvement_millage, new_revenue, taxable = model_split_rate_tax(
    df=taxable,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    current_revenue=taxable['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

exempt = gdf[gdf['full_exmp'] == 1].copy()
for col in ['new_tax', 'tax_change', 'tax_change_pct', 'taxable_land_value', 'taxable_improvement_value']:
    exempt[col] = 0.0
gdf = pd.concat([taxable, exempt]).sort_index()

print(f"Land millage:        {land_millage:.4f} mills")
print(f"Improvement millage: {improvement_millage:.4f} mills")
print(f"(Prior flat rate:    {CITY_MILLAGE:.3f} mills)")
print(f"Revenue check:       ${new_revenue:,.0f}")
print()

category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title="Reading, PA — 4:1 Split-Rate Tax Impact (LYCD Land Values)")

## Section 8 — Exploration Charts

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
cat_med = (
    gdf[gdf['PROPERTY_CATEGORY'] != 'Exempt']
    .groupby('PROPERTY_CATEGORY')['tax_change_pct']
    .median()
    .sort_values()
)
colors = ['#d62728' if v > 0 else '#2ca02c' for v in cat_med.values]
cat_med.plot.barh(ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Reading, PA — Median Tax Change % by Category (4:1 Split-Rate, LYCD Land Values)')
ax.set_xlabel('Median % Change')
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview_lycd.png', dpi=150)
plt.close()
print("Saved category_preview_lycd.png")

## Section 9 — Census Join + Export

In [ ]:
_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
        except concurrent.futures.TimeoutError:
            print("Census API timed out — skipping")
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f"Census join failed: {e}")
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

In [ ]:
from lvt.viz import create_city_report

out_df = save_standard_export(
    df=gdf,
    city=f'{CITY_NAME}_lycd',
    output_path=f'../../analysis/data/{CITY_NAME}_lycd.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)

create_city_report(out_df, f'{CITY_NAME}_lycd', show=False)
print("Done.")

## Validation Summary

| Check | Result |
|---|---|
| Revenue match | Same current tax base as model.ipynb (1994 CAMA, 19.217 mills) |
| LYCD land base | See Section 6 output (LYCD/CAMA ratio should be ~8–12× reflecting 1994→2026 appreciation) |
| AVM coverage | 99.7% of Reading parcels have AVM predictions; 79 parcels get KNN land values from neighbors |
| KNN window | K=75 nearest improved parcels; at ~6,150 improved parcels/sq mile, radius ≈ 330 ft |

**LYCD methodology note:** Land allocation is 20% of total AVM value for improved parcels and
100% for vacant parcels. This mirrors OPA's LYCD method (Philadelphia) adapted for Reading.
The 20% figure reflects a conservative land fraction consistent with Berks County's
historic assessment practice. A higher percentage (e.g. 30%) would shift more of the
tax burden to land and reduce taxes on improvements more aggressively.

**AVM quality note:** berks_open_avmkit residential_sf ensemble has Median Ratio=1.02,
COD=12.6% (IAAO-compliant). Commercial has only 1 test sale; those parcels' LYCD land
values derive mainly from nearby residential PSF via the KNN window, which may understate
commercial land values in the central business district.